In [1]:
import os
import torch
import numpy as np
from astropy.table import Table as aT
from astropy.cosmology import Planck13

In [2]:
import signal
from contextlib import contextmanager

In [3]:
from tqdm import tqdm

In [4]:
from sedflow import flows as F
from sedflow import galaxy as G

In [5]:
import corner as DFM
# --- plotting ---
import matplotlib as mpl
import matplotlib.pyplot as plt
#mpl.rcParams['text.usetex'] = True
#mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['axes.xmargin'] = 1
mpl.rcParams['xtick.labelsize'] = 'x-large'
mpl.rcParams['xtick.major.size'] = 5
mpl.rcParams['xtick.major.width'] = 1.5
mpl.rcParams['ytick.labelsize'] = 'x-large'
mpl.rcParams['ytick.major.size'] = 5
mpl.rcParams['ytick.major.width'] = 1.5
mpl.rcParams['legend.frameon'] = False

In [6]:
# read extinction corrected fluxes for BGS ANY
bgs = aT.read('/tigress/chhahn/sedflow/BGS_ANY_full.fluxes.hdf5')

In [7]:
ichunk = 0 
bgs = bgs[ichunk*100000:(ichunk+1)*100000]

In [8]:
bgs[:5]

TARGETID,Z_not4clus,Z_not4clus.mask,ZSUCCESS,ZSUCCESS.mask,FLUX_G_CORR,FLUX_R_CORR,FLUX_Z_CORR,FLUX_W1_CORR,FLUX_W2_CORR,FLUX_SIG_G,FLUX_SIG_R,FLUX_SIG_Z,FLUX_SIG_W1,FLUX_SIG_W2
int64,float64,bool,bool,bool,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32
39627322533347488,0.3933561693027759,False,True,False,6.308694,14.59653,26.823706,37.508904,34.790462,0.035606574,0.043013,0.09695761,0.6443017,1.3214793
39627322533349236,0.4318407392830489,False,True,False,4.1814938,11.420789,21.353018,30.502308,34.404987,0.03893559,0.05782043,0.1483977,0.66073304,1.3559905
39627322533350395,0.2740806294985723,False,True,False,8.085996,30.673317,63.10016,68.74799,48.995106,0.039037608,0.061913554,0.13528107,0.809526,1.548193
39627322533350460,nan,True,False,True,6.591036,18.135826,34.476997,44.125492,34.28704,0.029648302,0.04172614,0.09289031,0.69804883,1.3684057
39627322533350746,nan,True,False,True,4.4487734,11.42604,20.059301,26.682968,17.279203,0.039028414,0.055432074,0.14964703,0.66495466,1.3856004


## load `DESIflow`

In [9]:
if torch.cuda.is_available(): device = 'cuda'
else: device = 'cpu'

gsed = G.ModelB(name='modelb.lowz')
desiflow = F.DESIflow(gsed=gsed, device=device)

# deploy `DESIflow`

In [10]:
#################################################################### 
# timeout exception for out of distribution 
#################################################################### 
class TimeoutException(Exception): pass

@contextmanager
def time_limit(seconds):
    def signal_handler(signum, frame):
        raise TimeoutException("Timed out!")
    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)


In [11]:
bgs['SEDFLOW_SAMPLES'] = np.empty((len(bgs), 100, 15)) # 100 subsamples from the posterior
bgs['SEDFLOW_MAP'] = np.empty((len(bgs), 15)) # maximum a posteriori 
bgs['SEDFLOW_LOGMSTAR_SAMPLES'] = np.empty((len(bgs), 100))
bgs['SEDFLOW_LOGMSTAR_MAP'] = np.empty(len(bgs))

In [12]:
fluxes = np.array([bgs[col] for col in ['FLUX_G_CORR', 'FLUX_R_CORR', 'FLUX_Z_CORR', 'FLUX_W1_CORR', 'FLUX_W2_CORR']]).T
sig_fluxes = np.array([bgs[col] for col in ['FLUX_SIG_G', 'FLUX_SIG_R', 'FLUX_SIG_Z', 'FLUX_SIG_W1', 'FLUX_SIG_W2']]).T

In [18]:
n_sample = 10000
for igal in tqdm(np.arange(len(bgs))[bgs['ZSUCCESS']][:5]):   
    try:  
        with time_limit(10): 
            posterior_samples, logp = desiflow.run(fluxes[igal], sig_fluxes[igal], bgs['Z_not4clus'][igal], 
                                                       Nsample=n_sample, 
                                                       log_prob=True,
                                                       progress_bar=False)

        bgs['SEDFLOW_SAMPLES'][igal] = posterior_samples[::100] # randomly select 100 draws
        bgs['SEDFLOW_MAP'][igal] = posterior_samples[np.argmax(logp)] # determine MAP

        # calculate log M* (surviving mass)
        tage = Planck13.age(bgs['Z_not4clus'][igal]).value # age in Gyr
        bgs['SEDFLOW_LOGMSTAR_SAMPLES'][igal] = gsed._msurv(posterior_samples[::100], np.repeat(tage, 100))
        bgs['SEDFLOW_LOGMSTAR_MAP'][igal] = gsed._msurv(bgs['SEDFLOW_MAP'][igal], np.array([tage]))
    except TimeoutException: 
        print('TARGETID = %i TIMEDOUT' % (bgs['igal'][igal]))
        continue  


100%|██████████| 5/5 [00:01<00:00,  3.12it/s]


In [16]:
%timeit desiflow.run(fluxes[igal], sig_fluxes[igal], bgs['Z_not4clus'][igal], Nsample=10000, log_prob=True, progress_bar=False)

314 ms ± 174 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
